In [ ]:

#I chose to Recode the Checklist example using Open AI SDK

#  The imports

from rich.console import Console
import os
import requests
from dotenv import load_dotenv
from openai.types.responses import ResponseTextDeltaEvent
from agents import Agent, Runner, trace, function_tool, SQLiteSession, ModelSettings
load_dotenv(override=True)

In [ ]:
# Some lists!

checklist = []
completed = []

In [ ]:
def show(text):
    try:
        Console().print(text)
    except Exception:
        print(text)

In [ ]:

def get_checklist_report() -> str:
    result = ""
    for index, item in enumerate(checklist):
        if completed[index]:
            result += f"Checklist #{index + 1}: [green][strike]{item}[/strike][/green]\n"
        else:
            result += f"Checklist #{index + 1}: {item}\n"
    show(result)
    return result

In [ ]:
@function_tool
def create_checklist(descriptions: list[str]) -> str:
    """ Create a checklist to accomplish the task """
    checklist.extend(descriptions)
    completed.extend([False] * len(descriptions))
    return get_checklist_report()

In [ ]:
@function_tool
def mark_complete(index: int, completion_notes: str) -> str:
    """Mark a checklist item complete when it is done """
    if 1 <= index <= len(checklist):
        completed[index - 1] = True
    else:
        return "No checklist at this index."
    Console().print(completion_notes)
    return get_checklist_report()

In [ ]:
# agent with name, instructions, model

checklistagent = Agent(name="ChecklistAgent", model_settings=ModelSettings(parallel_tool_calls=False),
 instructions="""You are given a 
problem to solve, by using your checklist tools to plan a list of steps, then 
carrying out each step in turn. Now create a plan, set the checklist, carry out
 the steps, and reply with the solution. Make sure that you carry out each step
  one a time and mark it complete before moving on to the next step. You must
  call mark_complete for EVERY item, including the last one. Put the last step's
  work in completion_notes, mark it complete, and only then give your final reply.
  If any quantity isn't provided in the question, then include a step to come up with 
  a reasonable estimate. Provide your solution in Rich console markup without 
  code blocks. Do not ask the user questions or clarification; respond only with 
  the answer after using your tools.""", model="gpt-5.4-mini", tools=[create_checklist, mark_complete])

In [ ]:
checklist.clear()
completed.clear()

with trace("checklist agent"):
    result = await Runner.run(checklistagent, """Please act as an expert meal planner
     and budget coordinator. I need you to create a 7-day meal plan (Breakfast, Lunch,
     and Dinner) using a strict list of ingredients I already have, plus a maximum $50
     budget for any missing items.Here is what is currently in my fridge and pantry:4 
     chicken breasts, 1 block of cheddar cheese, 6 eggs, 1 bag of spinach, Half a bottle of 
     hot sauce, 1 bag of white rice, 2 onions, Olive oil, salt, and pepper. Constraints:Zero Waste: 
     Try to use up all the fresh ingredients listed above by the end of the week.Strict Budget: 
     You can add missing ingredients to make complete meals, but the total estimated 
     cost of the grocery list must be under $50.Efficiency: Group the required 
     additions into a single, organized grocery shopping list by supermarket aisle.
     Format your response with the 7-day meal schedule first, followed by the priced 
     grocery shopping list.""")

show(result.final_output)

# The model often puts the last step in final_output instead of calling mark_complete.
if checklist and not all(completed):
    for i, done in enumerate(completed):
        if not done:
            completed[i] = True
    get_checklist_report()